<a href="https://colab.research.google.com/github/sahanasb/Intelligent-E-Commerce-Search-with-Retrieval-Augmented-Generation/blob/main/BM25%2BSBERT%2BHybridSearch%2BRedis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas rank_bm25 sentence-transformers scikit-learn

In [2]:
!pip install langchain langchain-community langchain-groq sentence-transformers chromadb --break-system-packages

In [3]:
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import torch
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np
import shutil
import os

# loading data
df = pd.read_csv('/content/clean_data.csv')
df.head()

,name,main_category,image,ratings,no_of_ratings,actual_price,source_file,actual_price_usd,ranking_score,product_id,search_text
0,Aston Andia Women Waist Cinchers Corset Adjust...,Womens Innerwear,https://m.media-amazon.com/images/I/51qLMr3Fea...,4.0,43,999,Womens Innerwear.csv,10.99,3.68,0,product: aston andia women waist cinchers cors...
1,Floret Women's Cotton Non Padded Wire Free T-S...,Womens Innerwear,https://m.media-amazon.com/images/I/61sBzjDk7W...,4.0,1,399,Womens Innerwear.csv,4.39,2.91,1,product: floret women's cotton non padded wire...
2,DOT WAVE Women's Cotton Pyjamas - Loungepants ...,Womens Innerwear,https://m.media-amazon.com/images/I/81qQK0+n4p...,4.0,153,1699,Womens Innerwear.csv,18.69,3.89,2,product: dot wave women's cotton pyjamas - lou...
3,Bali womens Comfort Revolution Wireless T-shir...,Womens Innerwear,https://m.media-amazon.com/images/I/81ReKuvZQr...,4.0,9328,1500,Womens Innerwear.csv,16.50,4.25,3,product: bali womens comfort revolution wirele...
4,TRYLO Women's Non-Wired Bra,Womens Innerwear,https://m.media-amazon.com/images/W/IMAGERENDE...,4.0,11,435,Womens Innerwear.csv,4.78,3.43,4,product: trylo women's non-wired bra. category...


In [4]:
# Vector embedding
def build_doc(row, index):
    return Document(
        page_content=str(row['search_text']),
        metadata={
            # add index for Hybrid Search
            "idx": index,
            "name": str(row['name']),
            "category": str(row['main_category']),
            "price_usd": float(row['actual_price_usd']),
            "ratings": float(row['ratings'])
        }
    )
docs = [build_doc(row, index) for index, row in df.iterrows()]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"})

/tmp/ipykernel_55386/2127683449.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
if os.path.exists("./product_db"):
    shutil.rmtree("./product_db")
    print("Deleted old product_db")

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./product_db"
)

Deleted old product_db


In [6]:
results = vectorstore.similarity_search("i am planning to go to beach")

for i, doc in enumerate(results[:3]):
    print(f"\n--- Document {i+1} ---")
    print("Name:", doc.metadata.get("name"))
    print("Category:", doc.metadata.get("category"))
    print("Price:", doc.metadata.get("price_usd"))
    print("Rating:", doc.metadata.get("ratings"))
    print("Content:", doc.page_content)


--- Document 1 ---
Name: Beach Toy Set for Kids 8 Pcs with Castle Shaped Bucket Shovels Water Sprinkler and Moulds Made in India Perfect Beach Toy ...
Category: Toys
Price: 3.58
Rating: 4.4
Content: product: beach toy set for kids 8 pcs with castle shaped bucket shovels water sprinkler and moulds made in india perfect beach toy .... category: toys. price: $3.58. rating: 4.4 out of 5. reviews: 13. beach toy set for kids 8 pcs with castle shaped bucket shovels water sprinkler and moulds made in india perfect beach toy ...

--- Document 2 ---
Name: BEREAL Macaron -White Cork Sandal Women
Category: Womens Sandals
Price: 14.29
Rating: 5.0
Content: product: bereal macaron -white cork sandal women. category: womens sandals. price: $14.29. rating: 5.0 out of 5. reviews: 1. bereal macaron -white cork sandal women

--- Document 3 ---
Name: Inc.5 womens 1191_peach Sandal
Category: Womens Sandals
Price: 27.39
Rating: 5.0
Content: product: inc.5 womens 1191_peach sandal. category: womens sandals. 

#BM25

In [10]:
# Tokenization
tokenized_corpus = [doc.page_content.lower().split() for doc in docs]

# Initialization
bm25 = BM25Okapi(tokenized_corpus)

# BM25 query
def bm25_search(query, top_k=5):
  # Preprocess the user's query
  tokenized_query = query.lower().split()

  # Calculate scores for all documents in the corpus
  bm25_scores = bm25.get_scores(tokenized_query)

  # Get indices of top_k results
  top_indices = np.argsort(bm25_scores)[::-1][:top_k]
  print(f"Query: {query}")

  for i in top_indices:
      print(f"Score: {bm25_scores[i]:.2f} | Name: {df.iloc[i]['name']}")
  return top_indices, bm25_scores
# Test BM25
indices, scores = bm25_search("i am planning to go to beach")

Query: i am planning to go to beach
Score: 22.47 | Name: Hangout Hub Couple Men's & Women's Cotton Printed Regular Fit T-Shirts (Pack of 2) - I Am Her King I Am His Queen
Score: 19.71 | Name: Jungle Magic Doodle Waterz - Reusable I Water Colouring Book - Vehicles I Self-Drying with Easy to Hold Water Pen I Educat...
Score: 15.84 | Name: SanDisk Extreme 64GB CompactFlash Memory Card UDMA 7 Speed Up to 120MB/s & Extreme Pro SD UHS I 128GB Card for 4K Video fo...
Score: 15.54 | Name: E2B Adaptor/E-Type to B-Type Mount/elinchrom to Bowens/for Studio Light/Quick Lock/All-Metal/Spring-Loaded/elinchrom to godox
Score: 14.19 | Name: PARTY PROPZ Bachelorette Combo 1 Bride to BE Eye Glass, 1 Bride to BE SASH & 1 Set of Bride to BE PHOTOBOOTH /Bachelorette...


#SBERT

In [9]:
#SBERT
def sbert_search(query, top_k=5):
  # convert a single string of text into a vector
  query_embedding = embeddings.embed_query(query)

  # finds the closest matches using metrics like cosine similarity
  results = vectorstore.similarity_search(query, k=top_k)
  return results

# Test SBERT
query = "i am planning to go to beach"
sbert_results = sbert_search(query)
for doc in sbert_results:
    print(f"- {doc.metadata['name']}")

- Beach Toy Set for Kids 8 Pcs with Castle Shaped Bucket Shovels Water Sprinkler and Moulds Made in India Perfect Beach Toy ...
- BEREAL Macaron -White Cork Sandal Women
- Inc.5 womens 1191_peach Sandal
- Skechers Womens Go Walk 5 Foamies - Pup Life Sandal
- Skechers womens Max Cushioning Sandal


#Hybrid Search

In [12]:
def hybrid_search(query, top_k=5):
  # get BM25 rank
  tokenized_query = query.lower().split()
  bm25_scores = bm25.get_scores(tokenized_query)
  bm25_top_indices = np.argsort(bm25_scores)[::-1]
  bm25_mapping = {idx: rank for rank, idx in enumerate(bm25_top_indices)}

  # get SBERT rank
  sbert_results = vectorstore.similarity_search(query, k=100)

  sbert_mapping = {}
  for rank, doc in enumerate(sbert_results):
    idx = doc.metadata.get('idx')
    sbert_mapping[idx] = rank

  # Reciprocal Rank Fusion (RRF) to combine two ranked results
  rrf_scores = {}
  k = 60
  all_candidate_indices = set(list(bm25_top_indices[:100]) + list(sbert_mapping.keys()))

  for idx in all_candidate_indices:
    rank_bm = bm25_mapping.get(idx, 1000)
    rank_sbert = sbert_mapping.get(idx, 1000)

    score = (1 / (k + rank_bm)) + (1 / (k + rank_sbert))
    rrf_scores[idx] = score
  # sort score
  final_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
  return [docs[i] for i in final_indices]

# Test hybrid search
querty = "i am planning to go to beach"
hybrid_results = hybrid_search(query)
for doc in hybrid_results:
    print(f"- {doc.metadata['name']}")

- Beach Toy Set for Kids 8 Pcs with Castle Shaped Bucket Shovels Water Sprinkler and Moulds Made in India Perfect Beach Toy ...
- Skechers Womens Go Walk 5 Foamies - Pup Life Sandal
- BEREAL Macaron -White Cork Sandal Women
- Wildcraft 29.5 Ltrs Girl Squad 3 Beach Pink Casual Backpack (12352_Beach_Pink)(HxWxD : 17.5x12x8.5)(inches)
- WELCOME Women's Tan Sandal-6 Kids UK (LC 11)


# Redis

In [13]:
pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 2.5 MB/s eta 0:00:00


In [26]:
!apt-get install redis-server > /dev/null
!redis-server --daemonize yes

In [25]:
import redis
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
try:
  if r.ping():
    print("connect redis successfully")
except Exception as e:
    print("not connect redis")

connect redis successfully


In [23]:
import json
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
def final_search_pipeline(query, top_k=5):
  start_time = time.time()

  # check redis cache
  cache_key = f"search:{query.lower().strip()}"
  cached_res = r.get(cache_key)

  if cached_res:
    latency = (time.time() - start_time) * 1000
    print(f"CATCH HIT running: {latency:.2f}ms")
    return json.loads(cached_res)

  # query not in cache
  initial_candidates = hybrid_search(query, top_k=50)
  final_results = initial_candidates[:top_k]

  # TODO rerank_result then switch initial_candidates[:top_k] to rerank_results(query, initial_candidates, top_n=top_k)

  #final_results = rerank_results(query, initial_candidates, top_n=top_k)
  output = []
  for doc in final_results:
    output.append({
        "name": doc.metadata.get("name"),
        "price": doc.metadata.get("price_usd"),
        "category": doc.metadata.get("category"),
        "idx": doc.metadata.get("idx")
    })
  r.setex(cache_key, 3600, json.dumps(output))
  return output
results = final_search_pipeline("i am planning to go to beach")

🚀 [CACHE HIT] running: 0.88ms
